# 3H Final Fixed Notebook — Fast, Resumable, Journal-Safe

This notebook is designed to **stop the 3h experiment from wasting days**.

What is fixed:
- No duplicate code blocks.
- No moving-average window crash.
- No broken Transformer `Dropout` bug.
- Runs are **checkpointed** after every model, so if Colab disconnects, rerun and it skips completed models.
- Official selection uses **Validation TSS only**.
- Test set is used only for final reporting.
- Bootstrap CI and McNemar are key-aligned.

Default mode is **FAST_FINAL** to close the 3h chapter quickly.  
Change `RUN_MODE = "FULL_TUNING"` only if you really want to run the heavy full grid.


In [2]:
# ============================================================
# 0) MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# 1) SETUP
# ============================================================

import os
import re
import gc
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from scipy.stats import chi2

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)


TensorFlow: 2.20.0
Seed: 42


In [4]:
# ============================================================
# 2) PATHS + RUN MODE
# ============================================================

BASE = "/content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY"

DATA_PATH = f"{BASE}/HMI_AR_2010_2025_ML_READY_16_LABELED_MX3H_FLARE_FEATURES_HOURLY.csv"

OUT_DIR = f"{BASE}/3H_FINAL_FIXED_FAST_SAFE"
SPLIT_CACHE_DIR = os.path.join(OUT_DIR, "split_cache")
PRED_DIR = os.path.join(OUT_DIR, "predictions")
RESULTS_DIR = os.path.join(OUT_DIR, "results")

for d in [OUT_DIR, SPLIT_CACHE_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------
# FAST_FINAL is recommended to close the 3h chapter quickly.
# FULL_TUNING is available but heavy.
# ------------------------------------------------------------
RUN_MODE = "FAST_FINAL"   # "FAST_FINAL" or "FULL_TUNING"

MAX_GAP_HOURS = 2.0
EPOCHS_TUNE = 20
BOOT_N = 2000

# For extreme imbalance. Set to 1.0 if you want pure balanced class weights.
POS_MULTIPLIER = 1.15

print("RUN_MODE:", RUN_MODE)
print("Loading:", DATA_PATH)
print("Saving to:", OUT_DIR)


RUN_MODE: FAST_FINAL
Loading: /content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/HMI_AR_2010_2025_ML_READY_16_LABELED_MX3H_FLARE_FEATURES_HOURLY.csv
Saving to: /content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/3H_FINAL_FIXED_FAST_SAFE


In [5]:
# ============================================================
# 3) LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

df["T_REC_dt"] = pd.to_datetime(df["T_REC_dt"], errors="coerce")
df["NOAA_AR"] = pd.to_numeric(df["NOAA_AR"], errors="coerce").astype("Int64")
df["label_MX_3h"] = pd.to_numeric(df["label_MX_3h"], errors="coerce").fillna(0).astype(int)

MAG_FEATURES = [
    "MEANGBZ", "MEANGAM", "MEANGBT", "MEANGBH", "MEANJZD",
    "TOTUSJZ", "MEANALP", "MEANJZH", "ABSNJZH", "SAVNCPP",
    "MEANSHR", "SHRGT45", "R_VALUE", "USFLUX", "TOTPOT", "TOTUSJH"
]

FLARE_HISTORY_FEATURES = [
    "flare_count_past_3h",
    "time_since_last_flare_hours_3h",
    "max_peak_flux_past_3h",
    "mean_peak_flux_past_3h",
    "flare_activity_index_past_3h",
    "had_flare_past_3h",
]

ALL_FEATURES = MAG_FEATURES + FLARE_HISTORY_FEATURES

needed_cols = ["T_REC_dt", "NOAA_AR", "label_MX_3h"] + ALL_FEATURES
missing_cols = [c for c in needed_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# Basic imputation
for col in ALL_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    med = df[col].median()
    if pd.isna(med):
        med = 0.0
    df[col] = df[col].fillna(med)

df = df.dropna(subset=["T_REC_dt", "NOAA_AR"]).copy()
df["NOAA_AR"] = df["NOAA_AR"].astype(int)
df = df.sort_values(["NOAA_AR", "T_REC_dt"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Feature count:", len(ALL_FEATURES))
print(df["label_MX_3h"].value_counts())
print(df["label_MX_3h"].value_counts(normalize=True))

positive_rate = df["label_MX_3h"].mean()
print(f"3h positive rate / random PR-AUC baseline ≈ {positive_rate:.6f}")


Dataset shape: (482189, 27)
Feature count: 22
label_MX_3h
0    479138
1      3051
Name: count, dtype: int64
label_MX_3h
0    0.993673
1    0.006327
Name: proportion, dtype: float64
3h positive rate / random PR-AUC baseline ≈ 0.006327


In [6]:
# ============================================================
# 4) MODEL CANDIDATES
# ============================================================

if RUN_MODE == "FAST_FINAL":
    # This is the rescue mode. It runs a small but fair set:
    # all four model families, lookbacks 4 and 8.
    # Lookback 4 is included because previous partial runs showed strong 3h TSS.
    LOOKBACK_GRID = [4, 8]

    LSTM_CANDIDATES = [
        {"units": 64,  "dense_units": 32, "dropout": 0.2, "lr": 1e-3, "batch_size": 256},
        {"units": 128, "dense_units": 32, "dropout": 0.2, "lr": 5e-4, "batch_size": 128},
    ]

    BILSTM_CANDIDATES = [
        {"units": 128, "dense_units": 32, "dropout": 0.2, "lr": 5e-4, "batch_size": 128},
        {"units": 128, "dense_units": 32, "dropout": 0.2, "lr": 1e-3, "batch_size": 128},
        {"units": 64,  "dense_units": 32, "dropout": 0.3, "lr": 1e-3, "batch_size": 128},
    ]

    DLSTM_CANDIDATES = [
        {"ma_window": 3, "units": 64,  "dropout": 0.2, "lr": 1e-3, "batch_size": 256},
        {"ma_window": 3, "units": 128, "dropout": 0.2, "lr": 1e-3, "batch_size": 128},
    ]

    TRANSFORMER_CANDIDATES = [
        {"num_blocks": 1, "num_heads": 2, "key_dim": 16, "ff_dim": 64, "dropout": 0.2, "lr": 1e-3, "batch_size": 256},
        {"num_blocks": 1, "num_heads": 4, "key_dim": 16, "ff_dim": 64, "dropout": 0.2, "lr": 5e-4, "batch_size": 256},
    ]

else:
    # Heavy mode. Use only when you have time.
    LOOKBACK_GRID = [4, 8, 12]

    LSTM_CANDIDATES = [
        {"units": u, "dense_units": 32, "dropout": d, "lr": lr, "batch_size": bs}
        for u in [64, 128] for d in [0.2, 0.3] for lr in [1e-3, 5e-4] for bs in [128, 256]
    ]

    BILSTM_CANDIDATES = [
        {"units": u, "dense_units": 32, "dropout": d, "lr": lr, "batch_size": bs}
        for u in [64, 128] for d in [0.2, 0.3] for lr in [1e-3, 5e-4] for bs in [128, 256]
    ]

    DLSTM_CANDIDATES = [
        {"ma_window": ma, "units": u, "dropout": d, "lr": lr, "batch_size": bs}
        for ma in [3, 5] for u in [64, 128] for d in [0.2, 0.3] for lr in [1e-3, 5e-4] for bs in [128, 256]
    ]

    TRANSFORMER_CANDIDATES = [
        {"num_blocks": b, "num_heads": h, "key_dim": 16, "ff_dim": 64, "dropout": d, "lr": lr, "batch_size": bs}
        for b in [1, 2] for h in [2, 4] for d in [0.2, 0.3] for lr in [1e-3, 5e-4] for bs in [128, 256]
    ]

print("Lookbacks:", LOOKBACK_GRID)
print("Candidates per lookback:",
      "LSTM", len(LSTM_CANDIDATES),
      "BiLSTM", len(BILSTM_CANDIDATES),
      "DLSTM", len(DLSTM_CANDIDATES),
      "Transformer", len(TRANSFORMER_CANDIDATES))


Lookbacks: [4, 8]
Candidates per lookback: LSTM 2 BiLSTM 3 DLSTM 2 Transformer 2


In [7]:
# ============================================================
# 5) METRIC HELPERS
# ============================================================

def tss_hss_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn + 1e-9)
    fpr = fp / (fp + tn + 1e-9)
    tss = recall - fpr

    numerator = 2 * (tp * tn - fp * fn)
    denominator = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn) + 1e-9)
    hss = numerator / denominator

    return float(tss), float(hss)


def tune_threshold_by_tss(y_true, prob):
    thresholds = np.linspace(0, 1, 1001)
    best_thr = 0.5
    best_tss = -999.0

    for thr in thresholds:
        y_pred = (prob >= thr).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tss, _ = tss_hss_from_cm(cm)

        if tss > best_tss:
            best_tss = tss
            best_thr = thr

    return float(best_thr), float(best_tss)


def evaluate_probs(model_name, y_val, val_prob, y_test, test_prob, extra=None):
    threshold, val_tss = tune_threshold_by_tss(y_val, val_prob)

    y_pred = (test_prob >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tss, hss = tss_hss_from_cm(cm)
    tn, fp, fn, tp = cm.ravel()

    result = {
        "Model": model_name,
        "Threshold": threshold,
        "Val_TSS": val_tss,
        "Test_TSS": tss,
        "HSS": hss,
        "ROC_AUC": float(roc_auc_score(y_test, test_prob)),
        "PR_AUC": float(average_precision_score(y_test, test_prob)),
        "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "Precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "F1": float(f1_score(y_test, y_pred, zero_division=0)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

    if extra:
        result.update(extra)

    return result, y_pred


def get_class_weights(y, pos_multiplier=1.0):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    out = {int(c): float(w) for c, w in zip(classes, weights)}

    if 1 in out:
        out[1] *= float(pos_multiplier)

    return out


def mcnemar_test(y_true, pred_a, pred_b):
    y_true = np.asarray(y_true).astype(int)
    pred_a = np.asarray(pred_a).astype(int)
    pred_b = np.asarray(pred_b).astype(int)

    correct_a = pred_a == y_true
    correct_b = pred_b == y_true

    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    n = b + c

    if n == 0:
        chi2_stat = 0.0
        p_value = 1.0
    else:
        chi2_stat = ((abs(b - c) - 1) ** 2) / (n + 1e-9)
        p_value = float(chi2.sf(chi2_stat, df=1))

    return {"b": b, "c": c, "n": n, "chi2": float(chi2_stat), "p_value": p_value}


def bootstrap_tss_ci(y_true, y_pred, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    scores = []
    n = len(y_true)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue

        cm = confusion_matrix(y_true[idx], y_pred[idx], labels=[0, 1])
        tss, _ = tss_hss_from_cm(cm)
        scores.append(tss)

    scores = np.asarray(scores, dtype=float)

    return {
        "mean": float(np.mean(scores)),
        "ci_low": float(np.percentile(scores, 2.5)),
        "ci_high": float(np.percentile(scores, 97.5)),
        "n_boot_valid": int(len(scores)),
    }, scores


In [8]:
# ============================================================
# 6) SEQUENCE + SPLIT + CACHE HELPERS
# ============================================================

def safe_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))


def build_sequences_with_keys(frame, feature_cols, lookback, max_gap_hours=2.0):
    X_seq, y_seq, t_seq, ar_seq = [], [], [], []

    for ar_num, g in frame.groupby("NOAA_AR"):
        g = g.sort_values("T_REC_dt").reset_index(drop=True)

        if len(g) < lookback:
            continue

        vals = g[feature_cols].values.astype(np.float32)
        labels = g["label_MX_3h"].values.astype(int)
        times = pd.to_datetime(g["T_REC_dt"]).values

        for i in range(lookback - 1, len(g)):
            window_times = pd.to_datetime(times[i - lookback + 1:i + 1])
            diffs = pd.Series(window_times).diff().dropna().dt.total_seconds() / 3600.0

            if len(diffs) > 0 and (diffs > max_gap_hours).any():
                continue

            X_seq.append(vals[i - lookback + 1:i + 1])
            y_seq.append(labels[i])
            t_seq.append(pd.Timestamp(times[i]))
            ar_seq.append(int(ar_num))

    X_seq = np.asarray(X_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=int)

    keys = pd.DataFrame({
        "NOAA_AR": np.asarray(ar_seq, dtype=int),
        "T_REC_dt": pd.to_datetime(np.asarray(t_seq)),
        "y": y_seq,
    })

    return X_seq, y_seq, keys


def chronological_split_with_keys(X_seq, y_seq, keys_df, train_frac=0.70, val_frac=0.15):
    order = np.argsort(pd.to_datetime(keys_df["T_REC_dt"]).values)

    X_seq = X_seq[order]
    y_seq = y_seq[order]
    keys_df = keys_df.iloc[order].reset_index(drop=True)

    n = len(X_seq)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)

    return {
        "X_train": X_seq[:n_train],
        "y_train": y_seq[:n_train],
        "keys_train": keys_df.iloc[:n_train].reset_index(drop=True),

        "X_val": X_seq[n_train:n_train+n_val],
        "y_val": y_seq[n_train:n_train+n_val],
        "keys_val": keys_df.iloc[n_train:n_train+n_val].reset_index(drop=True),

        "X_test": X_seq[n_train+n_val:],
        "y_test": y_seq[n_train+n_val:],
        "keys_test": keys_df.iloc[n_train+n_val:].reset_index(drop=True),
    }


def scale_3d_train_only(X_train, X_val, X_test):
    scaler = StandardScaler()
    n_features = X_train.shape[-1]

    X_train_scaled = scaler.fit_transform(
        X_train.reshape(-1, n_features)
    ).reshape(X_train.shape)

    X_val_scaled = scaler.transform(
        X_val.reshape(-1, n_features)
    ).reshape(X_val.shape)

    X_test_scaled = scaler.transform(
        X_test.reshape(-1, n_features)
    ).reshape(X_test.shape)

    return X_train_scaled.astype(np.float32), X_val_scaled.astype(np.float32), X_test_scaled.astype(np.float32)


def build_or_load_split(lookback):
    npz_path = os.path.join(SPLIT_CACHE_DIR, f"split_lb{lookback}.npz")
    val_keys_path = os.path.join(SPLIT_CACHE_DIR, f"keys_val_lb{lookback}.csv")
    test_keys_path = os.path.join(SPLIT_CACHE_DIR, f"keys_test_lb{lookback}.csv")

    if os.path.exists(npz_path) and os.path.exists(val_keys_path) and os.path.exists(test_keys_path):
        print(f"Loading cached split for lookback={lookback}")
        data = np.load(npz_path, allow_pickle=False)
        return {
            "X_train_scaled": data["X_train_scaled"],
            "y_train": data["y_train"],
            "X_val_scaled": data["X_val_scaled"],
            "y_val": data["y_val"],
            "X_test_scaled": data["X_test_scaled"],
            "y_test": data["y_test"],
            "keys_val": pd.read_csv(val_keys_path, parse_dates=["T_REC_dt"]),
            "keys_test": pd.read_csv(test_keys_path, parse_dates=["T_REC_dt"]),
        }

    print("\n" + "=" * 72)
    print(f"Building sequences for lookback={lookback}")
    print("=" * 72)

    X_seq, y_seq, keys = build_sequences_with_keys(
        df,
        feature_cols=ALL_FEATURES,
        lookback=lookback,
        max_gap_hours=MAX_GAP_HOURS
    )

    split = chronological_split_with_keys(X_seq, y_seq, keys)
    X_train_sc, X_val_sc, X_test_sc = scale_3d_train_only(
        split["X_train"], split["X_val"], split["X_test"]
    )

    out = {
        "X_train_scaled": X_train_sc,
        "y_train": split["y_train"],
        "X_val_scaled": X_val_sc,
        "y_val": split["y_val"],
        "X_test_scaled": X_test_sc,
        "y_test": split["y_test"],
        "keys_val": split["keys_val"],
        "keys_test": split["keys_test"],
    }

    np.savez_compressed(
        npz_path,
        X_train_scaled=out["X_train_scaled"],
        y_train=out["y_train"],
        X_val_scaled=out["X_val_scaled"],
        y_val=out["y_val"],
        X_test_scaled=out["X_test_scaled"],
        y_test=out["y_test"],
    )

    out["keys_val"].to_csv(val_keys_path, index=False)
    out["keys_test"].to_csv(test_keys_path, index=False)

    print("Saved cached split:", npz_path)
    print("Train:", out["X_train_scaled"].shape, int(out["y_train"].sum()))
    print("Val  :", out["X_val_scaled"].shape, int(out["y_val"].sum()))
    print("Test :", out["X_test_scaled"].shape, int(out["y_test"].sum()))

    return out


def moving_average_decomposition(X, window=3):
    # Fast safe centered moving average over time dimension only.
    seq_len = X.shape[1]
    if window > seq_len:
        raise ValueError(f"MA window {window} cannot exceed lookback/sequence length {seq_len}")

    half = window // 2
    trend = np.empty_like(X, dtype=np.float32)

    for t in range(seq_len):
        start = max(0, t - half)
        end = min(seq_len, t + half + 1)
        trend[:, t, :] = X[:, start:end, :].mean(axis=1)

    residual = X - trend
    return trend.astype(np.float32), residual.astype(np.float32)


In [9]:
# ============================================================
# 7) MODEL BUILDERS
# ============================================================

def make_callbacks(patience=5):
    return [
        EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=0
        ),
        ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=max(2, patience // 2),
            min_lr=1e-6,
            verbose=0
        )
    ]


def build_lstm_model(input_shape, units=64, dense_units=32, dropout=0.2, lr=1e-3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(units, return_sequences=False, dropout=dropout, recurrent_dropout=0.0),
        layers.Dense(dense_units, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )
    return model


def build_bilstm_model(input_shape, units=64, dense_units=32, dropout=0.2, lr=1e-3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Bidirectional(
            layers.LSTM(units, return_sequences=False, dropout=dropout, recurrent_dropout=0.0)
        ),
        layers.Dense(dense_units, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )
    return model


def build_dlstm_model(input_shape, units=64, dropout=0.2, lr=1e-3):
    second_units = max(16, units // 2)
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(units, return_sequences=True, dropout=dropout, recurrent_dropout=0.0),
        layers.Dropout(dropout),
        layers.LSTM(second_units, return_sequences=False, dropout=dropout, recurrent_dropout=0.0),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )
    return model


def transformer_encoder_block(x, num_heads=2, key_dim=16, ff_dim=64, dropout=0.2):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout
    )(x, x)

    x = layers.Add()([x, attn])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(x.shape[-1])(ff)

    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x


def build_transformer_model(
    input_shape,
    num_blocks=1,
    num_heads=2,
    key_dim=16,
    ff_dim=64,
    dense_units=32,
    dropout=0.2,
    lr=1e-3
):
    inp = layers.Input(shape=input_shape)
    x = layers.Dense(64, activation="relu")(inp)

    for _ in range(num_blocks):
        x = transformer_encoder_block(
            x,
            num_heads=num_heads,
            key_dim=key_dim,
            ff_dim=ff_dim,
            dropout=dropout
        )

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(dropout)(x)

    out = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inp, out, name="Transformer_3h")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )
    return model


In [10]:
# ============================================================
# 8) TRAIN/EVALUATE ONE MODEL WITH CHECKPOINTING
# ============================================================

def save_prediction_frame(path, keys, y, prob, pred):
    frame = keys.copy()
    frame["y"] = np.asarray(y).astype(int)
    frame["prob"] = np.asarray(prob).astype(float)
    frame["pred"] = np.asarray(pred).astype(int)
    frame.to_csv(path, index=False)


def load_existing_results():
    result_files = [
        os.path.join(RESULTS_DIR, f)
        for f in os.listdir(RESULTS_DIR)
        if f.startswith("result__") and f.endswith(".json")
    ]

    rows = []
    for f in result_files:
        with open(f, "r") as handle:
            rows.append(json.load(handle))

    return rows


def train_evaluate_checkpointed(model_name, family, lookback, split, X_train, X_val, X_test, build_fn, build_kwargs, batch_size, extra):
    safe = safe_name(model_name)

    result_path = os.path.join(RESULTS_DIR, f"result__{safe}.json")
    val_pred_path = os.path.join(PRED_DIR, f"val__{safe}.csv")
    test_pred_path = os.path.join(PRED_DIR, f"test__{safe}.csv")
    error_path = os.path.join(RESULTS_DIR, f"error__{safe}.txt")

    if os.path.exists(result_path) and os.path.exists(val_pred_path) and os.path.exists(test_pred_path):
        print(f"SKIP existing: {model_name}")
        with open(result_path, "r") as handle:
            return json.load(handle)

    tf.keras.backend.clear_session()
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    print("\n" + "=" * 90)
    print("Training:", model_name)
    print("=" * 90)

    try:
        input_shape = (X_train.shape[1], X_train.shape[2])
        model = build_fn(input_shape=input_shape, **build_kwargs)

        class_weight = get_class_weights(split["y_train"], pos_multiplier=POS_MULTIPLIER)

        history = model.fit(
            X_train,
            split["y_train"],
            validation_data=(X_val, split["y_val"]),
            epochs=EPOCHS_TUNE,
            batch_size=batch_size,
            class_weight=class_weight,
            callbacks=make_callbacks(patience=5),
            shuffle=False,
            verbose=1
        )

        val_prob = model.predict(X_val, batch_size=512, verbose=0).ravel()
        test_prob = model.predict(X_test, batch_size=512, verbose=0).ravel()

        result, test_pred = evaluate_probs(
            model_name,
            split["y_val"], val_prob,
            split["y_test"], test_prob,
            extra={
                "Family": family,
                "Lookback": lookback,
                "Epochs_Used": len(history.history["loss"]),
                **extra
            }
        )

        _, val_pred = evaluate_probs(
            model_name + "_val_only",
            split["y_val"], val_prob,
            split["y_val"], val_prob,
            extra={}
        )

        save_prediction_frame(val_pred_path, split["keys_val"], split["y_val"], val_prob, val_pred)
        save_prediction_frame(test_pred_path, split["keys_test"], split["y_test"], test_prob, test_pred)

        with open(result_path, "w") as handle:
            json.dump(result, handle, indent=2)

        print(
            f"Val_TSS={result['Val_TSS']:.4f} | "
            f"Test_TSS={result['Test_TSS']:.4f} | "
            f"PR_AUC={result['PR_AUC']:.4f} | "
            f"Precision={result['Precision']:.4f} | "
            f"F1={result['F1']:.4f}"
        )

        del model
        gc.collect()
        return result

    except Exception as e:
        with open(error_path, "w") as handle:
            handle.write(repr(e))
        print("ERROR in", model_name, ":", repr(e))
        gc.collect()
        return None


In [11]:
# ============================================================
# 9) RUN FAST/CONTROLLED TUNING
# This cell can be rerun safely. Completed models are skipped.
# ============================================================

all_rows = load_existing_results()
completed_models = set([r["Model"] for r in all_rows])
print("Existing completed runs:", len(completed_models))

for lb in LOOKBACK_GRID:
    split = build_or_load_split(lb)

    # LSTM
    for cfg in LSTM_CANDIDATES:
        model_name = f"LSTM_3h_LB{lb}_U{cfg['units']}_D{cfg['dropout']}_LR{cfg['lr']}_BS{cfg['batch_size']}"
        if model_name in completed_models:
            continue

        result = train_evaluate_checkpointed(
            model_name=model_name,
            family="LSTM",
            lookback=lb,
            split=split,
            X_train=split["X_train_scaled"],
            X_val=split["X_val_scaled"],
            X_test=split["X_test_scaled"],
            build_fn=build_lstm_model,
            build_kwargs={
                "units": cfg["units"],
                "dense_units": cfg["dense_units"],
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
            },
            batch_size=cfg["batch_size"],
            extra={
                "MA_Window": np.nan,
                "Units": cfg["units"],
                "Dense_Units": cfg["dense_units"],
                "Dropout": cfg["dropout"],
                "Learning_Rate": cfg["lr"],
                "Batch_Size": cfg["batch_size"],
                "Transformer_Blocks": np.nan,
                "Transformer_Heads": np.nan,
                "Transformer_Key_Dim": np.nan,
                "Transformer_FF_Dim": np.nan,
            }
        )
        if result is not None:
            all_rows.append(result)
            completed_models.add(model_name)

    # BiLSTM
    for cfg in BILSTM_CANDIDATES:
        model_name = f"BiLSTM_3h_LB{lb}_U{cfg['units']}_D{cfg['dropout']}_LR{cfg['lr']}_BS{cfg['batch_size']}"
        if model_name in completed_models:
            continue

        result = train_evaluate_checkpointed(
            model_name=model_name,
            family="BiLSTM",
            lookback=lb,
            split=split,
            X_train=split["X_train_scaled"],
            X_val=split["X_val_scaled"],
            X_test=split["X_test_scaled"],
            build_fn=build_bilstm_model,
            build_kwargs={
                "units": cfg["units"],
                "dense_units": cfg["dense_units"],
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
            },
            batch_size=cfg["batch_size"],
            extra={
                "MA_Window": np.nan,
                "Units": cfg["units"],
                "Dense_Units": cfg["dense_units"],
                "Dropout": cfg["dropout"],
                "Learning_Rate": cfg["lr"],
                "Batch_Size": cfg["batch_size"],
                "Transformer_Blocks": np.nan,
                "Transformer_Heads": np.nan,
                "Transformer_Key_Dim": np.nan,
                "Transformer_FF_Dim": np.nan,
            }
        )
        if result is not None:
            all_rows.append(result)
            completed_models.add(model_name)

    # DLSTM
    for cfg in DLSTM_CANDIDATES:
        ma_window = cfg["ma_window"]
        if ma_window > lb:
            print(f"Skipping DLSTM lb={lb}, ma_window={ma_window}: MA window > lookback")
            continue

        model_name = f"DLSTM_3h_LB{lb}_MA{ma_window}_U{cfg['units']}_D{cfg['dropout']}_LR{cfg['lr']}_BS{cfg['batch_size']}"
        if model_name in completed_models:
            continue

        trend_train, res_train = moving_average_decomposition(split["X_train_scaled"], window=ma_window)
        trend_val, res_val = moving_average_decomposition(split["X_val_scaled"], window=ma_window)
        trend_test, res_test = moving_average_decomposition(split["X_test_scaled"], window=ma_window)

        X_train_d = np.concatenate([trend_train, res_train], axis=2)
        X_val_d = np.concatenate([trend_val, res_val], axis=2)
        X_test_d = np.concatenate([trend_test, res_test], axis=2)

        result = train_evaluate_checkpointed(
            model_name=model_name,
            family="DLSTM",
            lookback=lb,
            split=split,
            X_train=X_train_d,
            X_val=X_val_d,
            X_test=X_test_d,
            build_fn=build_dlstm_model,
            build_kwargs={
                "units": cfg["units"],
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
            },
            batch_size=cfg["batch_size"],
            extra={
                "MA_Window": ma_window,
                "Units": cfg["units"],
                "Dense_Units": np.nan,
                "Dropout": cfg["dropout"],
                "Learning_Rate": cfg["lr"],
                "Batch_Size": cfg["batch_size"],
                "Transformer_Blocks": np.nan,
                "Transformer_Heads": np.nan,
                "Transformer_Key_Dim": np.nan,
                "Transformer_FF_Dim": np.nan,
            }
        )
        if result is not None:
            all_rows.append(result)
            completed_models.add(model_name)

        del trend_train, res_train, trend_val, res_val, trend_test, res_test, X_train_d, X_val_d, X_test_d
        gc.collect()

    # Transformer
    for cfg in TRANSFORMER_CANDIDATES:
        model_name = (
            f"Transformer_3h_LB{lb}_B{cfg['num_blocks']}_H{cfg['num_heads']}"
            f"_K{cfg['key_dim']}_FF{cfg['ff_dim']}_D{cfg['dropout']}_LR{cfg['lr']}_BS{cfg['batch_size']}"
        )
        if model_name in completed_models:
            continue

        result = train_evaluate_checkpointed(
            model_name=model_name,
            family="Transformer",
            lookback=lb,
            split=split,
            X_train=split["X_train_scaled"],
            X_val=split["X_val_scaled"],
            X_test=split["X_test_scaled"],
            build_fn=build_transformer_model,
            build_kwargs={
                "num_blocks": cfg["num_blocks"],
                "num_heads": cfg["num_heads"],
                "key_dim": cfg["key_dim"],
                "ff_dim": cfg["ff_dim"],
                "dense_units": 32,
                "dropout": cfg["dropout"],
                "lr": cfg["lr"],
            },
            batch_size=cfg["batch_size"],
            extra={
                "MA_Window": np.nan,
                "Units": np.nan,
                "Dense_Units": 32,
                "Dropout": cfg["dropout"],
                "Learning_Rate": cfg["lr"],
                "Batch_Size": cfg["batch_size"],
                "Transformer_Blocks": cfg["num_blocks"],
                "Transformer_Heads": cfg["num_heads"],
                "Transformer_Key_Dim": cfg["key_dim"],
                "Transformer_FF_Dim": cfg["ff_dim"],
            }
        )
        if result is not None:
            all_rows.append(result)
            completed_models.add(model_name)

# Collate saved result JSONs again
all_rows = load_existing_results()
tuning_all_df = pd.DataFrame(all_rows)

if len(tuning_all_df) == 0:
    raise RuntimeError("No completed results found. Check error files in RESULTS_DIR.")

tuning_by_val = tuning_all_df.sort_values("Val_TSS", ascending=False).reset_index(drop=True)
tuning_by_test = tuning_all_df.sort_values("Test_TSS", ascending=False).reset_index(drop=True)

tuning_by_val.to_csv(os.path.join(OUT_DIR, "3H_FINAL_FIXED_TUNING_BY_VAL_TSS.csv"), index=False)
tuning_by_test.to_csv(os.path.join(OUT_DIR, "AUDIT_ONLY_3H_FINAL_FIXED_BY_TEST_TSS.csv"), index=False)

print("\nBEST CONFIGURATIONS BY VALIDATION TSS")
display(tuning_by_val.head(20).round(4))

print("\nAUDIT ONLY: TOP CONFIGURATIONS BY TEST TSS")
display(tuning_by_test.head(20).round(4))

official_best_model = tuning_by_val.iloc[0]["Model"]
print("\nOfficial best validation-selected model:", official_best_model)
display(pd.DataFrame([tuning_by_val.iloc[0]]).round(4))


Existing completed runs: 0

Building sequences for lookback=4
Saved cached split: /content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/3H_FINAL_FIXED_FAST_SAFE/split_cache/split_lb4.npz
Train: (312897, 4, 22) 1588
Val  : (67049, 4, 22) 936
Test : (67050, 4, 22) 334

Training: LSTM_3h_LB4_U64_D0.2_LR0.001_BS256
Epoch 1/20
1223/1223 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - auc: 0.9214 - loss: 0.3921 - val_auc: 0.9259 - val_loss: 0.6420 - learning_rate: 0.0010
Epoch 2/20
1223/1223 ━━━━━━━━━━━━━━━━━━━━ 16s 13ms/step - auc: 0.9348 - loss: 0.3551 - val_auc: 0.9283 - val_loss: 0.6220 - learning_rate: 0.0010
Epoch 3/20
1223/1223 ━━━━━━━━━━━━━━━━━━━━ 16s 13ms/step - auc: 0.9367 - loss: 0.3488 - val_auc: 0.9277 - val_loss: 0.6127 - learning_rate: 0.0010
Epoch 4/20
1223/1223 ━━━━━━━━━━━━━━━━━━━━ 16s 13ms/step - auc: 0.9401 - loss: 0.3411 - val_auc: 0.9286 - val_loss: 0.6111 - learning_rate: 0.0010
Epoch 5/20
1223/1223 ━━━━━━━━━━━━━━━━━━━━ 16s 13ms/step - auc: 0.9397 - loss: 0.3403 - val_auc: 0.9279 

,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,...,MA_Window,Units,Dense_Units,Dropout,Learning_Rate,Batch_Size,Transformer_Blocks,Transformer_Heads,Transformer_Key_Dim,Transformer_FF_Dim
0,DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256,0.666,0.7266,0.6997,0.0790,0.9361,0.2115,0.7810,0.0465,0.0878,...,3.0,64.0,NaN,0.2,0.0010,256,NaN,NaN,NaN,NaN
1,BiLSTM_3h_LB8_U128_D0.2_LR0.001_BS128,0.606,0.7257,0.7248,0.0538,0.9348,0.1847,0.8529,0.0327,0.0631,...,NaN,128.0,32.0,0.2,0.0010,128,NaN,NaN,NaN,NaN
2,Transformer_3h_LB8_B1_H2_K16_FF64_D0.2_LR0.001...,0.515,0.7254,0.7085,0.0368,0.9362,0.1994,0.8954,0.0238,0.0463,...,NaN,NaN,32.0,0.2,0.0010,256,1.0,2.0,16.0,64.0
3,LSTM_3h_LB8_U128_D0.2_LR0.0005_BS128,0.657,0.7253,0.7191,0.0611,0.9359,0.1984,0.8301,0.0366,0.0702,...,NaN,128.0,32.0,0.2,0.0005,128,NaN,NaN,NaN,NaN
4,DLSTM_3h_LB8_MA3_U128_D0.2_LR0.001_BS128,0.771,0.7239,0.7028,0.0715,0.9316,0.1983,0.7941,0.0424,0.0804,...,3.0,128.0,NaN,0.2,0.0010,128,NaN,NaN,NaN,NaN
5,BiLSTM_3h_LB4_U128_D0.2_LR0.0005_BS128,0.639,0.7234,0.7031,0.0461,0.9294,0.1914,0.8473,0.0286,0.0553,...,NaN,128.0,32.0,0.2,0.0005,128,NaN,NaN,NaN,NaN
6,BiLSTM_3h_LB4_U128_D0.2_LR0.001_BS128,0.637,0.7227,0.7049,0.0477,0.9305,0.1815,0.8443,0.0294,0.0569,...,NaN,128.0,32.0,0.2,0.0010,128,NaN,NaN,NaN,NaN
7,Transformer_3h_LB8_B1_H4_K16_FF64_D0.2_LR0.000...,0.628,0.7225,0.6910,0.0629,0.9358,0.1827,0.7941,0.0377,0.0719,...,NaN,NaN,32.0,0.2,0.0005,256,1.0,4.0,16.0,64.0
8,Transformer_3h_LB4_B1_H4_K16_FF64_D0.2_LR0.000...,0.676,0.7222,0.6826,0.0395,0.9257,0.1923,0.8473,0.0251,0.0488,...,NaN,NaN,32.0,0.2,0.0005,256,1.0,4.0,16.0,64.0
9,LSTM_3h_LB8_U64_D0.2_LR0.001_BS256,0.684,0.7217,0.7298,0.0624,0.9273,0.1995,0.8399,0.0373,0.0715,...,NaN,64.0,32.0,0.2,0.0010,256,NaN,NaN,NaN,NaN



AUDIT ONLY: TOP CONFIGURATIONS BY TEST TSS


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,...,MA_Window,Units,Dense_Units,Dropout,Learning_Rate,Batch_Size,Transformer_Blocks,Transformer_Heads,Transformer_Key_Dim,Transformer_FF_Dim
0,LSTM_3h_LB8_U64_D0.2_LR0.001_BS256,0.684,0.7217,0.7298,0.0624,0.9273,0.1995,0.8399,0.0373,0.0715,...,NaN,64.0,32.0,0.2,0.0010,256,NaN,NaN,NaN,NaN
1,BiLSTM_3h_LB8_U128_D0.2_LR0.001_BS128,0.606,0.7257,0.7248,0.0538,0.9348,0.1847,0.8529,0.0327,0.0631,...,NaN,128.0,32.0,0.2,0.0010,128,NaN,NaN,NaN,NaN
2,LSTM_3h_LB8_U128_D0.2_LR0.0005_BS128,0.657,0.7253,0.7191,0.0611,0.9359,0.1984,0.8301,0.0366,0.0702,...,NaN,128.0,32.0,0.2,0.0005,128,NaN,NaN,NaN,NaN
3,BiLSTM_3h_LB8_U64_D0.3_LR0.001_BS128,0.717,0.7201,0.7154,0.0680,0.9315,0.1904,0.8137,0.0404,0.0770,...,NaN,64.0,32.0,0.3,0.0010,128,NaN,NaN,NaN,NaN
4,Transformer_3h_LB8_B1_H2_K16_FF64_D0.2_LR0.001...,0.515,0.7254,0.7085,0.0368,0.9362,0.1994,0.8954,0.0238,0.0463,...,NaN,NaN,32.0,0.2,0.0010,256,1.0,2.0,16.0,64.0
5,LSTM_3h_LB4_U64_D0.2_LR0.001_BS256,0.662,0.7088,0.7054,0.0521,0.9275,0.1673,0.8323,0.0318,0.0612,...,NaN,64.0,32.0,0.2,0.0010,256,NaN,NaN,NaN,NaN
6,BiLSTM_3h_LB4_U128_D0.2_LR0.001_BS128,0.637,0.7227,0.7049,0.0477,0.9305,0.1815,0.8443,0.0294,0.0569,...,NaN,128.0,32.0,0.2,0.0010,128,NaN,NaN,NaN,NaN
7,BiLSTM_3h_LB4_U128_D0.2_LR0.0005_BS128,0.639,0.7234,0.7031,0.0461,0.9294,0.1914,0.8473,0.0286,0.0553,...,NaN,128.0,32.0,0.2,0.0005,128,NaN,NaN,NaN,NaN
8,DLSTM_3h_LB8_MA3_U128_D0.2_LR0.001_BS128,0.771,0.7239,0.7028,0.0715,0.9316,0.1983,0.7941,0.0424,0.0804,...,3.0,128.0,NaN,0.2,0.0010,128,NaN,NaN,NaN,NaN
9,BiLSTM_3h_LB8_U128_D0.2_LR0.0005_BS128,0.686,0.7201,0.7008,0.0602,0.9316,0.2007,0.8105,0.0362,0.0693,...,NaN,128.0,32.0,0.2,0.0005,128,NaN,NaN,NaN,NaN



Official best validation-selected model: DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256


,Model,Threshold,Val_TSS,Test_TSS,HSS,ROC_AUC,PR_AUC,Recall,Precision,F1,...,MA_Window,Units,Dense_Units,Dropout,Learning_Rate,Batch_Size,Transformer_Blocks,Transformer_Heads,Transformer_Key_Dim,Transformer_FF_Dim
0,DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256,0.666,0.7266,0.6997,0.079,0.9361,0.2115,0.781,0.0465,0.0878,...,3.0,64.0,NaN,0.2,0.001,256,NaN,NaN,NaN,NaN


In [12]:
# ============================================================
# 10) FAMILY WINNERS + MCNEMAR WITH KEY ALIGNMENT
# ============================================================

def read_pred_frame(model_name, split_name):
    safe = safe_name(model_name)
    path = os.path.join(PRED_DIR, f"{split_name}__{safe}.csv")
    return pd.read_csv(path, parse_dates=["T_REC_dt"])


best_family_rows = (
    tuning_by_val
    .sort_values("Val_TSS", ascending=False)
    .groupby("Family", as_index=False)
    .first()
    .sort_values("Val_TSS", ascending=False)
    .reset_index(drop=True)
)

print("Best validation-selected model per family:")
display(best_family_rows[[
    "Family", "Model", "Lookback", "Val_TSS", "Test_TSS",
    "ROC_AUC", "PR_AUC", "Recall", "Precision", "F1"
]].round(4))

official_best_model = tuning_by_val.iloc[0]["Model"]
official_family = tuning_by_val.iloc[0]["Family"]

official_test = read_pred_frame(official_best_model, "test").rename(
    columns={"prob": "prob_official", "pred": "pred_official"}
)

mcnemar_rows = []

for _, row in best_family_rows.iterrows():
    comparator_model = row["Model"]
    comparator_family = row["Family"]

    if comparator_model == official_best_model:
        continue

    comp_test = read_pred_frame(comparator_model, "test").rename(
        columns={"prob": "prob_comp", "pred": "pred_comp"}
    )

    merged = official_test.merge(
        comp_test[["NOAA_AR", "T_REC_dt", "y", "prob_comp", "pred_comp"]],
        on=["NOAA_AR", "T_REC_dt", "y"],
        how="inner"
    )

    if len(merged) == 0:
        print("No overlapping test keys for", comparator_model)
        continue

    out = mcnemar_test(
        merged["y"].values,
        merged["pred_official"].values,
        merged["pred_comp"].values
    )

    out.update({
        "Comparison": f"Tuned Best {official_family} vs Best Tuned {comparator_family}",
        "Official_Best_Model": official_best_model,
        "Comparator_Model": comparator_model,
        "Comparator_Family": comparator_family,
        "Overlap_N": int(len(merged)),
        "Interpretation": "Significant difference" if out["p_value"] < 0.05 else "No significant difference"
    })
    mcnemar_rows.append(out)

mcnemar_df = pd.DataFrame(mcnemar_rows)

if len(mcnemar_df):
    mcnemar_df = mcnemar_df[[
        "Comparison",
        "Official_Best_Model",
        "Comparator_Model",
        "Comparator_Family",
        "Overlap_N",
        "b", "c", "n", "chi2", "p_value", "Interpretation"
    ]]
    display(mcnemar_df.round(6))
    mcnemar_df.to_csv(os.path.join(OUT_DIR, "MCNEMAR_3H_FINAL_FIXED_KEY_ALIGNED.csv"), index=False)
else:
    print("No McNemar comparisons available.")


Best validation-selected model per family:


,Family,Model,Lookback,Val_TSS,Test_TSS,ROC_AUC,PR_AUC,Recall,Precision,F1
0,DLSTM,DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256,8,0.7266,0.6997,0.9361,0.2115,0.7810,0.0465,0.0878
1,BiLSTM,BiLSTM_3h_LB8_U128_D0.2_LR0.001_BS128,8,0.7257,0.7248,0.9348,0.1847,0.8529,0.0327,0.0631
2,Transformer,Transformer_3h_LB8_B1_H2_K16_FF64_D0.2_LR0.001...,8,0.7254,0.7085,0.9362,0.1994,0.8954,0.0238,0.0463
3,LSTM,LSTM_3h_LB8_U128_D0.2_LR0.0005_BS128,8,0.7253,0.7191,0.9359,0.1984,0.8301,0.0366,0.0702


,Comparison,Official_Best_Model,Comparator_Model,Comparator_Family,Overlap_N,b,c,n,chi2,p_value,Interpretation
0,Tuned Best DLSTM vs Best Tuned BiLSTM,DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256,BiLSTM_3h_LB8_U128_D0.2_LR0.001_BS128,BiLSTM,64411,3127,315,3442,2295.677223,0.0,Significant difference
1,Tuned Best DLSTM vs Best Tuned Transformer,DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256,Transformer_3h_LB8_B1_H2_K16_FF64_D0.2_LR0.001...,Transformer,64411,6632,158,6790,6170.799558,0.0,Significant difference
2,Tuned Best DLSTM vs Best Tuned LSTM,DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256,LSTM_3h_LB8_U128_D0.2_LR0.0005_BS128,LSTM,64411,2042,254,2296,1390.840157,0.0,Significant difference


In [ ]:
# ============================================================
# 11) BOOTSTRAP CI FOR OFFICIAL BEST MODEL
# ============================================================

official_best_model = tuning_by_val.iloc[0]["Model"]
official_test = read_pred_frame(official_best_model, "test")

bootstrap_3h_ci, boot_scores = bootstrap_tss_ci(
    official_test["y"].values,
    official_test["pred"].values,
    n_boot=BOOT_N,
    seed=SEED
)

bootstrap_3h_ci["model"] = official_best_model
bootstrap_3h_ci["point_tss"] = float(tuning_by_val.iloc[0]["Test_TSS"])

print("Official tuned 3h model:", official_best_model)
print("Bootstrap TSS CI:", bootstrap_3h_ci)

pd.DataFrame([bootstrap_3h_ci]).to_csv(
    os.path.join(OUT_DIR, "BOOTSTRAP_3H_FINAL_FIXED_BEST_TSS_CI.csv"),
    index=False
)

np.save(
    os.path.join(OUT_DIR, "BOOTSTRAP_3H_FINAL_FIXED_BEST_TSS_SCORES.npy"),
    boot_scores
)


Official tuned 3h model: DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256
Bootstrap TSS CI: {'mean': 0.6987887496359394, 'ci_low': 0.6526116188540577, 'ci_high': 0.7421132200017929, 'n_boot_valid': 2000, 'model': 'DLSTM_3h_LB8_MA3_U64_D0.2_LR0.001_BS256', 'point_tss': 0.6996853701663841}


In [ ]:
# ============================================================
# 12) OPTIONAL FAMILY META-ENSEMBLE WITH KEY ALIGNMENT
# ============================================================

family_models = best_family_rows["Model"].tolist()

val_meta = None
test_meta = None

for model_name in family_models:
    family = tuning_by_val.loc[tuning_by_val["Model"] == model_name, "Family"].iloc[0]

    v = read_pred_frame(model_name, "val")[["NOAA_AR", "T_REC_dt", "y", "prob"]].rename(columns={"prob": f"prob_{family}"})
    t = read_pred_frame(model_name, "test")[["NOAA_AR", "T_REC_dt", "y", "prob"]].rename(columns={"prob": f"prob_{family}"})

    if val_meta is None:
        val_meta = v
        test_meta = t
    else:
        val_meta = val_meta.merge(v, on=["NOAA_AR", "T_REC_dt", "y"], how="inner")
        test_meta = test_meta.merge(t, on=["NOAA_AR", "T_REC_dt", "y"], how="inner")

print("Aligned validation meta:", val_meta.shape)
print("Aligned test meta:", test_meta.shape)

prob_cols = [c for c in val_meta.columns if c.startswith("prob_")]

meta_rows = []

if len(prob_cols) >= 2 and len(val_meta) > 0 and len(test_meta) > 0:
    X_meta_val = val_meta[prob_cols].values
    X_meta_test = test_meta[prob_cols].values
    y_meta_val = val_meta["y"].values.astype(int)
    y_meta_test = test_meta["y"].values.astype(int)

    # Average
    val_avg = X_meta_val.mean(axis=1)
    test_avg = X_meta_test.mean(axis=1)

    avg_result, avg_pred = evaluate_probs(
        "3H_FamilyMeta_Average",
        y_meta_val, val_avg,
        y_meta_test, test_avg,
        extra={"Family": "MetaEnsemble", "Lookback": "key_aligned", "Inputs": ", ".join(prob_cols)}
    )
    meta_rows.append(avg_result)

    # Logistic stacking with CV on aligned validation set
    if y_meta_val.sum() > 10:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        stacker = LogisticRegressionCV(
            Cs=10,
            cv=cv,
            class_weight="balanced",
            max_iter=3000,
            scoring="roc_auc"
        )
        stacker.fit(X_meta_val, y_meta_val)

        val_stack = stacker.predict_proba(X_meta_val)[:, 1]
        test_stack = stacker.predict_proba(X_meta_test)[:, 1]

        stack_result, stack_pred = evaluate_probs(
            "3H_FamilyMeta_Stacking",
            y_meta_val, val_stack,
            y_meta_test, test_stack,
            extra={"Family": "MetaEnsemble", "Lookback": "key_aligned", "Inputs": ", ".join(prob_cols)}
        )
        meta_rows.append(stack_result)

    meta_df = pd.DataFrame(meta_rows).sort_values("Val_TSS", ascending=False).reset_index(drop=True)
    display(meta_df.round(4))
    meta_df.to_csv(os.path.join(OUT_DIR, "3H_FAMILY_META_ENSEMBLE_KEY_ALIGNED.csv"), index=False)
else:
    print("Meta-ensemble skipped: not enough aligned models/keys.")


In [ ]:
# ============================================================
# 13) MANUSCRIPT-READY OUTPUTS
# ============================================================

family_winners_display = best_family_rows[[
    "Family", "Model", "Lookback", "Threshold", "Val_TSS", "Test_TSS",
    "HSS", "ROC_AUC", "PR_AUC", "Recall", "Precision", "F1",
    "TN", "FP", "FN", "TP"
]].copy()

family_winners_display = family_winners_display.sort_values("Val_TSS", ascending=False).reset_index(drop=True)

display(family_winners_display.round(4))

family_winners_display.to_csv(
    os.path.join(OUT_DIR, "MANUSCRIPT_READY_3H_FINAL_FIXED_FAMILY_WINNERS.csv"),
    index=False
)

print("Done. Use these files:")
print("1) 3H_FINAL_FIXED_TUNING_BY_VAL_TSS.csv")
print("2) AUDIT_ONLY_3H_FINAL_FIXED_BY_TEST_TSS.csv")
print("3) MANUSCRIPT_READY_3H_FINAL_FIXED_FAMILY_WINNERS.csv")
print("4) MCNEMAR_3H_FINAL_FIXED_KEY_ALIGNED.csv")
print("5) BOOTSTRAP_3H_FINAL_FIXED_BEST_TSS_CI.csv")
print("6) 3H_FAMILY_META_ENSEMBLE_KEY_ALIGNED.csv")
print("\nRecommended manuscript rule: use the official best by Validation TSS, not the audit-by-test ranking.")
